<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/rf_crosscheck_830794_zebra_top10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RF cross-check — our pipeline on the top-10 Zebra-correlated units (ecephys 830794)

Reproduces the stacked receptive-field figure in the README's *Temporary — RF cross-check* block, so the plots can be confirmed as the output of **this repo's** RF pipeline. Same session (sub-830794, 2026-01-26), the same 10 units, in the same top-to-bottom order as the external "top-10 by Zebra correlation" figure.

**Scope.** The 10 units were *selected* by an external Zebra-correlation analysis; this notebook takes those 10 unit tags as given and renders **our** receptive fields for them — it does not recompute the Zebra ranking.

**What our RF map is.** For each unit we bin spike rate by the Gabor patch's azimuth/elevation position (response window 30–200 ms minus a pre-stimulus baseline), then fit a 2-D Gaussian. The RF-mapping block presents patches at **three orientations (0°, 45°, 90°)** with everything else fixed, and our map **pools all presentations at each position** — i.e. it is the **orientation-averaged** spatial RF. (The external figure instead splits the three orientations into separate columns.)

**Two-file provenance.** The external unit tags are the first 8 characters of the unit UUID in the **AIND "v2" packaging** (2026-04-23); DANDI 001637 re-mints those UUIDs, so the tags do not survive into the DANDI file. We therefore read spikes + stimulus from the AIND file and borrow the CCF anatomy (which the AIND file predates) from the DANDI file by matching probe + extremum channel — the two describe the same physical probes (480 electrodes each).

In [ ]:
#@title Install dependencies
!pip -q install h5py remfile requests numpy scipy matplotlib

In [ ]:
#@title Streaming helpers (AIND public S3 + DANDI redirect)
import h5py, remfile, requests, numpy as np
from scipy.optimize import curve_fit

def dec(x): return x.decode() if isinstance(x,(bytes,bytearray)) else x

def open_aind(key):
    """Open a public AIND NWB (HDF5) for lazy HTTP reads."""
    url = f"https://aind-open-data.s3.amazonaws.com/{key}"
    return h5py.File(remfile.File(url), "r")

def open_dandi(asset_id, dandiset="001637", version="draft"):
    """Resolve a DANDI asset to its (signed, expiring) S3 URL and open it."""
    r = requests.get(
        f"https://api.dandiarchive.org/api/dandisets/{dandiset}/versions/{version}/assets/{asset_id}/download/",
        allow_redirects=False, timeout=30)
    return h5py.File(remfile.File(r.headers["Location"]), "r")

def col(group, name):
    return np.array([dec(x) for x in group[name][:]])

print("helpers ready")

In [ ]:
#@title Configuration — session, files, and the 10 units
# AIND "v2" packaging (2026-04-23): the unit UUIDs whose first-8 chars are the external tags.
AIND_KEY = "ecephys_830794_2026-01-26_12-02-05_nwb_2026-04-23_18-12-45/ecephys_830794_2026-01-26_12-02-05_experiment1_recording1.nwb"
# DANDI 001637 asset for the same session — source of the CCF anatomy.
DANDI_ASSET = "0766996b-2afa-4c6f-a381-134c9e380edc"
# The 10 units, in the external figure's top-to-bottom order (first-8 of the AIND unit UUID):
HEX_TAGS = ["39e51e22","abb8e295","f4e75445","08ec65d2","bfcdaaa2",
            "444da816","6e34faa3","656a9b39","e8377ab6","5a003096"]
# RF response / baseline windows (s), matching the repo RF pipeline:
RESP, BASE = (0.03, 0.20), (-0.15, -0.02)
print(len(HEX_TAGS), "units;", "RESP", RESP, "BASE", BASE)

In [ ]:
#@title Compute our RF maps from the AIND RF-mapping (Gabor-patch) block
fh = open_aind(AIND_KEY)
g = fh["intervals"]["RF mapping_presentations"]
X = col(g,"X").astype(float); Y = col(g,"Y").astype(float); t0 = g["start_time"][:]
xs = np.array(sorted(set(X))); ys = np.array(sorted(set(Y)))
xi = {v:k for k,v in enumerate(xs)}; yi = {v:k for k,v in enumerate(ys)}
tX = np.array([xi[v] for v in X]); tY = np.array([yi[v] for v in Y])
oris = sorted(set(col(g,"Orientation").astype(float)))
print(f"RF grid {len(xs)}x{len(ys)}, {len(t0)} trials, orientations pooled: "
      f"{[round(np.degrees(o)) for o in oris]} deg")

U = fh["units"]
un = col(U,"unit_name"); first8 = np.array([u.replace("-","")[:8] for u in un])
dev = col(U,"device_name"); depth = U["depth"][:]; eci = U["extremum_channel_index"][:]
sti = U["spike_times_index"][:]; st = U["spike_times"]

def rf_map(i):
    lo = 0 if i==0 else int(sti[i-1]); hi = int(sti[i]); sp = st[lo:hi]
    grid = np.zeros((len(ys),len(xs))); cnt = np.zeros_like(grid); base = 0.0
    for k in range(len(t0)):
        s = t0[k]
        a = np.searchsorted(sp,[s+RESP[0],s+RESP[1]]); grid[tY[k],tX[k]] += (a[1]-a[0])/(RESP[1]-RESP[0])
        b = np.searchsorted(sp,[s+BASE[0],s+BASE[1]]); base += (b[1]-b[0])/(BASE[1]-BASE[0])
        cnt[tY[k],tX[k]] += 1
    return grid/np.clip(cnt,1,None) - base/len(t0)

units = {}
for h in HEX_TAGS:
    i = int(np.where(first8==h)[0][0])
    units[h] = dict(row=i, unit_name=un[i], probe=dev[i], depth=int(round(depth[i])),
                    ch=int(eci[i]), rf=rf_map(i))
fh.close()
print("computed", len(units), "RF maps")

In [ ]:
#@title Borrow CCF anatomy from DANDI (probe + extremum channel) and fit Gaussians
fd = open_dandi(DANDI_ASSET)
E = fd["general/extracellular_ephys/electrodes"]
e_loc = col(E,"location"); e_grp = col(E,"group_name")
offset = {g_:int(np.where(e_grp==g_)[0][0]) for g_ in set(e_grp)}
blocklen = {g_:int((e_grp==g_).sum()) for g_ in set(e_grp)}
fd.close()

XX, YY = np.meshgrid(xs, ys)
def fit2d(m):
    mm = m if abs(m.max())>=abs(m.min()) else -m
    p0 = [mm.max(), XX.ravel()[mm.argmax()], YY.ravel()[mm.argmax()], 12,12, np.median(mm)]
    edge = xs.max()
    try:
        popt,_ = curve_fit(lambda c,a,x0,y0,sx,sy,o:(a*np.exp(-((c[0]-x0)**2/(2*sx**2)+(c[1]-y0)**2/(2*sy**2)))+o).ravel(),
            (XX,YY), mm.ravel(), p0=p0, maxfev=6000,
            bounds=([0,-edge-20,-edge-20,3,3,-abs(mm).max()],[np.inf,edge+20,edge+20,80,80,abs(mm).max()]))
        fitv = popt[0]*np.exp(-((XX-popt[1])**2/(2*popt[3]**2)+(YY-popt[2])**2/(2*popt[4]**2)))+popt[5]
        return 2.355*np.sqrt(popt[3]*popt[4]), 1-np.sum((mm-fitv)**2)/np.sum((mm-mm.mean())**2)
    except Exception:
        return np.nan, np.nan

for h,u in units.items():
    g_ = u["probe"]
    u["area"] = e_loc[offset[g_] + min(u["ch"], blocklen[g_]-1)]
    u["fwhm"], u["r2"] = fit2d(u["rf"])
for h in HEX_TAGS:
    u = units[h]; print(f"{u['probe']} {h}  {u['area']:8s} d{u['depth']:>4}  FWHM {u['fwhm']:.0f} deg  R2={u['r2']:.2f}")

In [ ]:
#@title Render the stacked figure (matched order)
import matplotlib.pyplot as plt
probe_full = {"ProbeC":"ProbeC","ProbeD":"ProbeD"}
fig, axes = plt.subplots(len(HEX_TAGS), 1, figsize=(4.8, 22))
for ax,h in zip(axes, HEX_TAGS):
    u = units[h]; m = u["rf"]; vmax = np.abs(m).max()
    im = ax.imshow(m, origin="lower", extent=[xs[0],xs[-1],ys[0],ys[-1]], cmap="RdBu_r",
                   vmin=-vmax, vmax=vmax, aspect="equal")
    lvl = 0.5*m.max() if abs(m.max())>=abs(m.min()) else 0.5*m.min()
    ax.contour(XX, YY, m, levels=[lvl], colors="k", linewidths=0.8)
    ax.text(-0.42,0.5, f"{probe_full.get(u['probe'],u['probe'])}\n{h}", transform=ax.transAxes,
            ha="center", va="center", fontsize=8.5)
    ax.text(0.5,1.04, f"{u['area']} · d{u['depth']}µm", transform=ax.transAxes,
            ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.text(0.5,-0.17, f"FWHM {u['fwhm']:.0f}° · R²={u['r2']:.2f} · peak {m.max():.0f} Hz",
            transform=ax.transAxes, ha="center", va="top", fontsize=7, color="#555")
    ax.axhline(0,color="gray",lw=0.3); ax.axvline(0,color="gray",lw=0.3)
    ax.set_xticks([-40,0,40]); ax.set_yticks([-40,0,40]); ax.tick_params(labelsize=6.5)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=5.5)
axes[-1].set_xlabel("azimuth (°)  ·  elevation on y", fontsize=8, labelpad=20)
fig.suptitle("ecephys 830794 — our RF pipeline (same 10 units, matched order)", fontsize=11, y=0.997)
fig.subplots_adjust(left=0.23, right=0.85, top=0.975, bottom=0.035, hspace=0.55)
fig.savefig("../figures/our_rf_stacked_830794.png", dpi=200, bbox_inches="tight")
plt.show()

## Notes

- **Orientation averaging.** Each map pools the 0°/45°/90° Gabor presentations at every position, so it is the orientation-averaged spatial RF — good for RF *location* and *size*, but it does not resolve orientation tuning (that is what the external figure's three Gabor columns show).
- **Anatomy.** All 10 units resolve to the lateral geniculate (ProbeC → dorsal LGN `LGd-co`/`LGd-ip`; ProbeD → ventral LGN `LGv`) — expected when ranking by raw visual-stimulus correlation, which favours the most reliably driven relay cells. `depth` is µm from the probe tip.
- **Sign.** Red = ON (patch drives the cell), blue = OFF/suppression; several units are OFF-dominant to Gabor patches despite being Zebra-positive.